In [1]:
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
import lightgbm as lgb
import random
from sklearn.metrics import f1_score
from matplotlib import pyplot as plt
from sklearn.cluster import KMeans
import json
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import csr_matrix, coo_matrix
from scipy.sparse.linalg import svds
from scipy.spatial.distance import cosine
from sklearn.decomposition import NMF

from catboost import CatBoostClassifier

In [63]:
version = "v5"

In [3]:
orders = pd.read_csv("orders.csv")
vendors = pd.read_csv("vendors.csv")
train_customers = pd.read_csv("train_customers.csv")
train_locations = pd.read_csv("train_locations.csv")
test_customers = pd.read_csv("test_customers.csv")
test_locations = pd.read_csv("test_locations.csv")
sample_submission = pd.read_csv("SampleSubmission.csv")
validation_k_folds = pd.read_csv("validation_7_folds.csv")

/home/ubuntu/anaconda3/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3063: DtypeWarning: Columns (15,16,18,19,20) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


In [4]:
vendors['primary_tags'] = vendors['primary_tags'].fillna('{"primary_tags":"0"}').map(lambda x : 
                                                                                     int(json.loads(x)['primary_tags']))
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(vendors.vendor_tag.astype(str).map(lambda x : " ".join(x.split(","))))
U, sigma, Vt = svds(X, k = 8)
vendor_tags = pd.DataFrame(U)
vendor_tags.columns = ["vendor_tags_svd_" + str(c) for c in vendor_tags.columns]
vendors = pd.concat([vendors, vendor_tags], axis = 1)

In [5]:
max_created_date = pd.to_datetime(vendors.created_at).max()
member_since = (max_created_date - pd.to_datetime(vendors[['id', 'created_at']].set_index('id').created_at)).dt.days.to_dict()
vendors['vendor_member_since'] = vendors['id'].map(member_since)

In [6]:
for var in tqdm(['sunday_from_time1', 'sunday_to_time1',
       'sunday_from_time2', 'sunday_to_time2', 'monday_from_time1',
       'monday_to_time1', 'monday_from_time2', 'monday_to_time2',
       'tuesday_from_time1', 'tuesday_to_time1', 'tuesday_from_time2',
       'tuesday_to_time2', 'wednesday_from_time1', 'wednesday_to_time1',
       'wednesday_from_time2', 'wednesday_to_time2', 'thursday_from_time1',
       'thursday_to_time1', 'thursday_from_time2', 'thursday_to_time2',
       'friday_from_time1', 'friday_to_time1', 'friday_from_time2',
       'friday_to_time2', 'saturday_from_time1', 'saturday_to_time1',
       'saturday_from_time2', 'saturday_to_time2']):

    vendors[var] = pd.to_datetime(vendors[var], format = "%H:%M:%S").map(lambda x : x.hour + x.minute/60)

In [7]:
train_locations['key'] = 1
test_locations['key'] = 1
vendors['key'] = 1

target = orders[['customer_id', 'LOCATION_NUMBER', 'vendor_id']].drop_duplicates().rename(
                                                                columns = {'LOCATION_NUMBER' : 'location_number'})
target['target'] = 1
train = train_locations.merge(vendors.rename(columns = {'id' : 'vendor_id', 'latitude' : 'vendor_latitude', 
                        'longitude' : 'vendor_longitude'}), on = 'key', how = 'inner').merge(
                target, on = ['customer_id', 'location_number', 'vendor_id'], how = 'left').drop('key', axis = 1)
test = test_locations.merge(vendors.rename(columns = {'id' : 'vendor_id', 'latitude' : 'vendor_latitude', 
                                'longitude' : 'vendor_longitude'}), on = 'key', how = 'inner').drop('key', axis = 1)

train.target.fillna(0, inplace = True)
train.shape, test.shape

((5950300, 74), (1672000, 73))

In [8]:
customer_is_female = train_customers[['akeed_customer_id', 'gender']].append(test_customers[['akeed_customer_id', 'gender']]
                ).set_index('akeed_customer_id').gender.astype(str).map(lambda x : 1 if "female" in x.lower() else 0).to_dict()
train['customer_is_female'] = train['customer_id'].map(customer_is_female)
test['customer_is_female'] = test['customer_id'].map(customer_is_female)


In [9]:
customer_age = train_customers[['akeed_customer_id', 'dob']].append(test_customers[['akeed_customer_id', 'dob']]
                ).set_index('akeed_customer_id').dob.map(lambda x : 2020 - x).to_dict()
train['customer_age'] = train['customer_id'].map(customer_age)
test['customer_age'] = test['customer_id'].map(customer_age)


In [10]:
max_created_date = pd.to_datetime(train_customers.created_at.append(test_customers.created_at)).max()
member_since = (max_created_date - pd.to_datetime(train_customers[['akeed_customer_id', 'created_at']].append(test_customers[['akeed_customer_id', 'created_at']]
                ).set_index('akeed_customer_id').created_at)).dt.days.to_dict()
train['member_since'] = train['customer_id'].map(member_since)
test['member_since'] = test['customer_id'].map(member_since)


In [11]:
customer_num_updates = train_customers.akeed_customer_id.append(test_customers.akeed_customer_id).value_counts().to_dict()
train['customer_num_updates'] = train.customer_id.map(customer_num_updates)
test['customer_num_updates'] = test.customer_id.map(customer_num_updates)

In [12]:
def correct_lat(x):
    if x < -1000:
        return x  + 1038
    if x < -660:
        return x + 668
    if x < -500:
        return x + 535
    if x  < -150:
        return x + 154
    if x < -90:
        return x + 96
    if x < -50:
        return x + 49
    if x < -30:
        return x + 31
    if x < -55:
        return x + 60
    if x < -55:
        return x + 60
    if x > 8 and x < 16:
        return x - 12.5
    if x > 16 and x < 30:
        return x - 19
    if x > 90 and x < 100:
        return x - 97
    if x > 120 and x < 130:
        return x - 125.5
    if x > 150 and x < 160:
        return x - 155
    if x > 200 and x < 210:
        return x - 204
    if x > 280 and x < 300:
        return x - 284
    if x > 450 and x < 460:
        return x - 453
    return x
    
    
def correct_lon(x):
    if x < -130:
        return x  + 132
    if x < -100:
        return x + 102
    if x < -75:
        return x + 78
    if x  < -65:
        return x + 67
    if x < -55:
        return x + 60
    if x < -7:
        return x + 9
    if x < -2:
        return x + 3
    if x > 10 and x < 50:
        return x - 44
    return x


In [13]:
df = train_locations[train_locations.latitude.notnull()][['customer_id', 'location_number', 'latitude', 'longitude']]
kmeans = KMeans(n_clusters=16, random_state=0).fit(df[['latitude', 'longitude']].values)
df = train_locations[train_locations.latitude.notnull()][['customer_id', 'location_number', 'latitude', 'longitude']].append(
    test_locations[test_locations.latitude.notnull()][['customer_id', 'location_number', 'latitude', 'longitude']])
df['location_cluster'] = kmeans.predict(df[['latitude', 'longitude']].values)

In [14]:
location_clusters = df.set_index(['customer_id', 'location_number']).location_cluster.to_dict()
train['location_cluster'] = [location_clusters.get((row.customer_id, row.location_number), -1) for row in train.itertuples()]
test['location_cluster'] = [location_clusters.get((row.customer_id, row.location_number), -1) for row in test.itertuples()]

In [15]:
orders['location_cluster'] = [location_clusters.get((row.customer_id, row.LOCATION_NUMBER), -1) for row in orders.itertuples()]
vendor_location_cluster_order_count = orders[['vendor_id', 'location_cluster', 'akeed_order_id']].groupby(
                                        ['vendor_id', 'location_cluster']).count().akeed_order_id.to_dict()
train['vendor_location_cluster_order_count'] = [vendor_location_cluster_order_count.get((row.vendor_id, row.location_cluster), 0)
                                                for row in train.itertuples()]
test['vendor_location_cluster_order_count'] = [vendor_location_cluster_order_count.get((row.vendor_id, row.location_cluster), 0)
                                                for row in test.itertuples()]


In [16]:
train['longitude_corrected'] = train.longitude.map(correct_lon)
train['latitude_corrected'] = train.latitude.map(correct_lon)
test['longitude_corrected'] = test.longitude.map(correct_lon)
test['latitude_corrected'] = test.latitude.map(correct_lon)


train['vendor_longitude_corrected'] = train.vendor_longitude.map(correct_lon)
train['vendor_latitude_corrected'] = train.vendor_latitude.map(correct_lon)
test['vendor_longitude_corrected'] = test.vendor_longitude.map(correct_lon)
test['vendor_latitude_corrected'] = test.vendor_latitude.map(correct_lon)


In [17]:
def get_distance(lon1, lat1, lon2, lat2):
    return np.sqrt(((lat1 - lat2)**2 + (lon1 - lon2)**2)/2)

train['vendor_distance'] = [get_distance(row.longitude, row.latitude, row.vendor_longitude, row.vendor_latitude)
                           for row in tqdm(train.itertuples())]
test['vendor_distance'] = [get_distance(row.longitude, row.latitude, row.vendor_longitude, row.vendor_latitude)
                           for row in tqdm(test.itertuples())]

train['vendor_distance_corrected'] = [get_distance(row.longitude_corrected, row.latitude_corrected, row.vendor_longitude_corrected, row.vendor_latitude_corrected)
                           for row in tqdm(train.itertuples())]
test['vendor_distance_corrected'] = [get_distance(row.longitude_corrected, row.latitude_corrected, row.vendor_longitude_corrected, row.vendor_latitude_corrected)
                           for row in tqdm(test.itertuples())]

In [18]:
train['lon_diff'] = train['longitude'] - train['vendor_longitude']
test['lon_diff'] = test['longitude'] - test['vendor_longitude']


train['lat_diff'] = train['latitude'] - train['vendor_latitude']
test['lat_diff'] = test['latitude'] - test['vendor_latitude']


train['lon_diff_corrected'] = train['longitude_corrected'] - train['vendor_longitude_corrected']
test['lon_diff_corrected'] = test['longitude_corrected'] - test['vendor_longitude_corrected']


train['lat_diff_corrected'] = train['latitude_corrected'] - train['vendor_latitude_corrected']
test['lat_diff_corrected'] = test['latitude_corrected'] - test['vendor_latitude_corrected']


In [19]:
train['lon_diff_abs'] = abs(train['longitude'] - train['vendor_longitude'])
test['lon_diff_abs'] = abs(test['longitude'] - test['vendor_longitude'])

train['lat_diff_abs'] = abs(train['latitude'] - train['vendor_latitude'])
test['lat_diff_abs'] = abs(test['latitude'] - test['vendor_latitude'])


train['lon_diff_corrected_abs'] = abs(train['longitude_corrected'] - train['vendor_longitude_corrected'])
test['lon_diff_corrected_abs'] = abs(test['longitude_corrected'] - test['vendor_longitude_corrected'])


train['lat_diff_corrected_abs'] = abs(train['latitude_corrected'] - train['vendor_latitude_corrected'])
test['lat_diff_corrected_abs'] = abs(test['latitude_corrected'] - test['vendor_latitude_corrected'])


In [20]:
customer_location_counts = train_locations['customer_id'].append(test_locations['customer_id']).value_counts().to_dict()
train['customer_location_counts'] = train['customer_id'].map(customer_location_counts)
test['customer_location_counts'] = test['customer_id'].map(customer_location_counts)

In [21]:
nearby_radius = 0.25
num_vendors_nearby = train[train.vendor_distance < nearby_radius][['customer_id', 'location_number', 'vendor_id']].append(
                    test[test.vendor_distance < nearby_radius][['customer_id', 'location_number', 'vendor_id']]).groupby(
                    ['customer_id', 'location_number']).count().vendor_id.to_dict()
train['num_vendors_nearby'] = [num_vendors_nearby.get((row.customer_id, row.location_number), 0) for row in train.itertuples()]
test['num_vendors_nearby'] = [num_vendors_nearby.get((row.customer_id, row.location_number), 0) for row in test.itertuples()]


In [22]:
distance_rank = train[['customer_id', 'location_number', 'vendor_id', 'vendor_distance']].append(
                    test[['customer_id', 'location_number', 'vendor_id', 'vendor_distance']])
distance_rank['distance_rank'] = distance_rank[['customer_id', 'location_number', 'vendor_distance']
                                              ].groupby(['customer_id', 'location_number']).rank(method = 'min')
distance_rank = distance_rank.set_index(['customer_id', 'location_number', 'vendor_id']).distance_rank.to_dict()
train['distance_rank'] = [distance_rank.get((row.customer_id, row.location_number, row.vendor_id), 0) for row in train.itertuples()]
test['distance_rank'] = [distance_rank.get((row.customer_id, row.location_number, row.vendor_id), 0) for row in test.itertuples()]

In [23]:
distance_rank_lat = train[['customer_id', 'location_number', 'vendor_id', 'lat_diff_abs']].append(
                    test[['customer_id', 'location_number', 'vendor_id', 'lat_diff_abs']])
distance_rank_lat['distance_rank_lat'] = distance_rank_lat[['customer_id', 'location_number', 'lat_diff_abs']
                                              ].groupby(['customer_id', 'location_number']).rank(method = 'min')
distance_rank_lat = distance_rank_lat.set_index(['customer_id', 'location_number', 'vendor_id']).distance_rank_lat.to_dict()
train['distance_rank_lat'] = [distance_rank_lat.get((row.customer_id, row.location_number, row.vendor_id), 0) for row in train.itertuples()]
test['distance_rank_lat'] = [distance_rank_lat.get((row.customer_id, row.location_number, row.vendor_id), 0) for row in test.itertuples()]

In [24]:
distance_rank_lon = train[['customer_id', 'location_number', 'vendor_id', 'lon_diff_abs']].append(
                    test[['customer_id', 'location_number', 'vendor_id', 'lon_diff_abs']])
distance_rank_lon['distance_rank_lon'] = distance_rank_lon[['customer_id', 'location_number', 'lon_diff_abs']
                                              ].groupby(['customer_id', 'location_number']).rank(method = 'min')
distance_rank_lon = distance_rank_lon.set_index(['customer_id', 'location_number', 'vendor_id']).distance_rank_lon.to_dict()
train['distance_rank_lon'] = [distance_rank_lon.get((row.customer_id, row.location_number, row.vendor_id), 0) for row in train.itertuples()]
test['distance_rank_lon'] = [distance_rank_lon.get((row.customer_id, row.location_number, row.vendor_id), 0) for row in test.itertuples()]

In [25]:
vendor_uniuqe_customer_count = orders[['vendor_id', 'customer_id']].drop_duplicates().vendor_id.value_counts().to_dict()
train['vendor_uniuqe_customer_count'] = train['vendor_id'].map(vendor_uniuqe_customer_count)
test['vendor_uniuqe_customer_count'] = test['vendor_id'].map(vendor_uniuqe_customer_count)

In [26]:
vendor_uniuqe_customer_count = orders[['vendor_id', 'customer_id']].drop_duplicates().vendor_id.value_counts().to_dict()
train['vendor_uniuqe_customer_count'] = train['vendor_id'].map(vendor_uniuqe_customer_count)
test['vendor_uniuqe_customer_count'] = test['vendor_id'].map(vendor_uniuqe_customer_count)

In [27]:
df = orders[['vendor_id', 'customer_id', 'akeed_order_id']].groupby(['vendor_id', 'customer_id']).count().reset_index()
df['total_transaction'] = df[['vendor_id', 'akeed_order_id']].groupby('vendor_id').transform(sum)
df['customer_transaction_fraction'] = df['akeed_order_id'] / df['total_transaction']
df['plogp'] = -df.customer_transaction_fraction * df.customer_transaction_fraction.map(np.log)
vendor_entropy = df[['vendor_id', 'plogp']].groupby('vendor_id').sum().plogp.to_dict()

train['vendor_entropy'] = train.vendor_id.map(vendor_entropy)
test['vendor_entropy'] = test.vendor_id.map(vendor_entropy)


/home/ubuntu/anaconda3/lib/python3.7/site-packages/pandas/core/base.py:1154: RuntimeWarning: divide by zero encountered in log
  new_values = map_f(values, mapper)


In [28]:
vendor_total_amount_by_location_type = orders[['vendor_id', 'LOCATION_TYPE', 'grand_total']].fillna("nan").groupby(
                                    ['vendor_id', 'LOCATION_TYPE']).sum().grand_total.to_dict()
train['vendor_total_amount_by_location_type'] = [vendor_total_amount_by_location_type.get((row.vendor_id, str(row.location_type)), 0) for row in train.itertuples()]
test['vendor_total_amount_by_location_type'] = [vendor_total_amount_by_location_type.get((row.vendor_id, str(row.location_type)), 0) for row in test.itertuples()]

In [29]:
orders.delivery_date = pd.to_datetime(orders.delivery_date).dt.date
daily_order_counts = orders[['vendor_id', 'delivery_date', 'akeed_order_id']].groupby(['vendor_id', 'delivery_date']
                        ).count().reset_index()[['vendor_id', 'akeed_order_id']
                        ].groupby('vendor_id').agg(['mean', 'std']).akeed_order_id.to_dict()
train['daily_order_counts_mean'] = train.vendor_id.map(daily_order_counts['mean'])
train['daily_order_counts_std'] = train.vendor_id.map(daily_order_counts['std'])

test['daily_order_counts_mean'] = test.vendor_id.map(daily_order_counts['mean'])
test['daily_order_counts_std'] = test.vendor_id.map(daily_order_counts['std'])


In [30]:
vendor_ratings = orders[['vendor_id', 'vendor_rating']].groupby('vendor_id').agg(['mean', 'std', 'count']).vendor_rating.to_dict()
train['vendor_ratings_mean'] = train.vendor_id.map(vendor_ratings['mean'])
train['vendor_ratings_std'] = train.vendor_id.map(vendor_ratings['std'])
train['vendor_ratings_count'] = train.vendor_id.map(vendor_ratings['count'])

test['vendor_ratings_mean'] = test.vendor_id.map(vendor_ratings['mean'])
test['vendor_ratings_std'] = test.vendor_id.map(vendor_ratings['std'])
test['vendor_ratings_count'] = test.vendor_id.map(vendor_ratings['count'])


In [31]:
vendor_driver_ratings = orders[['vendor_id', 'driver_rating']].groupby('vendor_id').agg(['mean', 'std', 'count']).driver_rating.to_dict()
train['vendor_driver_ratings_mean'] = train.vendor_id.map(vendor_driver_ratings['mean'])
train['vendor_driver_ratings_std'] = train.vendor_id.map(vendor_driver_ratings['std'])
train['vendor_driver_ratings_count'] = train.vendor_id.map(vendor_driver_ratings['count'])

test['vendor_driver_ratings_mean'] = test.vendor_id.map(vendor_driver_ratings['mean'])
test['vendor_driver_ratings_std'] = test.vendor_id.map(vendor_driver_ratings['std'])
test['vendor_driver_ratings_count'] = test.vendor_id.map(vendor_driver_ratings['count'])


In [32]:
train['vendor_is_restaurant'] = train.vendor_category_en.map(lambda x : 1 if x == 'Restaurants' else 0)
test['vendor_is_restaurant'] = test.vendor_category_en.map(lambda x : 1 if x == 'Restaurants' else 0)

In [33]:
train.location_type.value_counts()
train['location_type_is_home'] = (train['location_type'] == "Home")*1
train['location_type_is_work'] = (train['location_type'] == "Work")*1
train['location_type_is_other'] = (train['location_type'] == "Other")*1

test['location_type_is_home'] = (test['location_type'] == "Home")*1
test['location_type_is_work'] = (test['location_type'] == "Work")*1
test['location_type_is_other'] = (test['location_type'] == "Other")*1


In [34]:
orders_delivery_distances = orders[['vendor_id', 'deliverydistance']].groupby('vendor_id').agg(['mean', 'std', 'max']).deliverydistance.to_dict()
train['vendor_mean_delivery_distance'] = train.vendor_id.map(orders_delivery_distances['mean'])
train['vendor_std_delivery_distance'] = train.vendor_id.map(orders_delivery_distances['std'])
train['vendor_max_delivery_distance'] = train.vendor_id.map(orders_delivery_distances['max'])


test['vendor_mean_delivery_distance'] = test.vendor_id.map(orders_delivery_distances['mean'])
test['vendor_std_delivery_distance'] = test.vendor_id.map(orders_delivery_distances['std'])
test['vendor_max_delivery_distance'] = test.vendor_id.map(orders_delivery_distances['max'])


In [35]:
cust_location_lat_range = (train_locations[['customer_id', 'latitude']].append(test_locations[['customer_id', 'latitude']]
                                ).groupby('customer_id').max() - train_locations[['customer_id', 'latitude']].append(
                                test_locations[['customer_id', 'latitude']]).groupby('customer_id').min()).latitude.map(abs).to_dict()
cust_location_lon_range = (train_locations[['customer_id', 'longitude']].append(test_locations[['customer_id', 'longitude']]
                                ).groupby('customer_id').max() - train_locations[['customer_id', 'longitude']].append(
                                test_locations[['customer_id', 'longitude']]).groupby('customer_id').min()).longitude.map(abs).to_dict()
train['cust_location_lat_range'] = train.customer_id.map(cust_location_lat_range)
test['cust_location_lat_range'] = test.customer_id.map(cust_location_lat_range)
train['cust_location_lon_range'] = train.customer_id.map(cust_location_lon_range)
test['cust_location_lon_range'] = test.customer_id.map(cust_location_lon_range)

In [36]:
vendor_location_lat_range = (orders.merge(train_locations[['customer_id', 'latitude']], on = 'customer_id')[['vendor_id', 'latitude']]
                            .groupby('vendor_id').max() - orders.merge(train_locations[['customer_id', 'latitude']], 
                            on = 'customer_id')[['vendor_id', 'latitude']].groupby('vendor_id').min()).latitude.map(abs).to_dict()
vendor_location_lon_range = (orders.merge(train_locations[['customer_id', 'longitude']], on = 'customer_id')[['vendor_id', 'longitude']].groupby('vendor_id').max() - orders.merge(train_locations[['customer_id', 'longitude']], 
                            on = 'customer_id')[['vendor_id', 'longitude']].groupby('vendor_id').min()).longitude.map(abs).to_dict()
train['vendor_location_lat_range'] = train.vendor_id.map(vendor_location_lat_range)
test['vendor_location_lat_range'] = test.vendor_id.map(vendor_location_lat_range)
train['vendor_location_lon_range'] = train.vendor_id.map(vendor_location_lon_range)
test['vendor_location_lon_range'] = test.vendor_id.map(vendor_location_lon_range)

In [37]:
train['member_since_diff'] = train['member_since'] - train['vendor_member_since']
test['member_since_diff'] = test['member_since'] - test['vendor_member_since']

In [38]:
df = orders[['vendor_id', 'customer_id', 'akeed_order_id']].sort_values(by = ['vendor_id', 'customer_id', 'akeed_order_id'])
df['prev_order_id'] = df.groupby(['vendor_id', 'customer_id']).shift(1)
df['prev_order_id_diff'] =  df['akeed_order_id'] - df['prev_order_id']
vendor_order_frequency = df[['vendor_id', 'prev_order_id_diff']].groupby('vendor_id').mean().prev_order_id_diff.to_dict()
train['vendor_order_frequency'] = train['vendor_id'].map(vendor_order_frequency)
test['vendor_order_frequency'] = test['vendor_id'].map(vendor_order_frequency)

repeat_customers = df[df.prev_order_id.isnull()==False].drop_duplicates()['vendor_id'].value_counts().to_dict()
train['repeat_customers'] = train['vendor_id'].map(repeat_customers)
test['repeat_customers'] = test['vendor_id'].map(repeat_customers)


In [39]:
def get_latent_features(df, u,v, key, dim = 16):
    """
    u = feature whose latent vector is to be calculate
    u = feature against whole the latent vector is calculated
    t = time var
    """
    df = df.sort_values(by = [u,v])
    df['c'] = 1    
    R_df = df[[u,v,'c']].groupby([u,v]).sum().reset_index()
    R_df.columns = [u, v, 'tf']
    n = df[u].nunique()
    R_df['idf'] = R_df[[u, v]].groupby(v).transform('count')[u].map(lambda x : np.log(n/x))
    R_df['tf_idf'] = R_df['tf']*R_df['idf']
#     R_df['tf_idf'] = R_df['tf_idf'] - R_df[[u, 'tf_idf']].groupby(u).transform('mean').tf_idf
    
    u_ids = sorted(df[u].unique())
    v_ids = sorted(df[v].unique())
    u_dict = {el : i for i, el in enumerate(u_ids)}
    v_dict = {el : i for i, el in enumerate(v_ids)}

    R_df['u_id'] = R_df[u].map(u_dict)
    R_df['v_id'] = R_df[v].map(v_dict)
    rows = R_df.u_id.values
    cols = R_df.v_id.values
    data = R_df.tf_idf.values
    R = coo_matrix((data, (rows, cols)), shape=(len(u_ids), len(v_ids)))
#     U, sigma, Vt = svds(R, k = dim)
    model = NMF(n_components=dim, init = "random", random_state=0)
    U = model.fit_transform(R)
    Vt = model.components_
    vendor_latent_features = pd.DataFrame(Vt).T
    vendor_latent_features.columns = ['{}_latent_feature_{}'.format(key,i) for i in range(vendor_latent_features.shape[1])]
    vendor_latent_features[v] = vendor_latent_features.index.map(lambda x : v_ids[x])
    return vendor_latent_features

vendor_latent_features = get_latent_features(orders, 'customer_id', 'vendor_id', 'vendor', 8)


In [40]:
train = train.merge(vendor_latent_features, on = 'vendor_id')
test = test.merge(vendor_latent_features, on = 'vendor_id')

In [41]:
df = train_locations[['customer_id', 'latitude', 'longitude']].append(test_locations[['customer_id', 'latitude', 'longitude']])
df.latitude = df.latitude.map(lambda x : np.round(x,2))
df.longitude = df.longitude.map(lambda x : np.round(x,2))
df['num_cust_on_location'] = df[['latitude', 'longitude', 'customer_id']].groupby(['latitude', 'longitude']).transform('count')
num_cust_on_location = df.set_index('customer_id').num_cust_on_location.to_dict()
train['num_cust_on_location'] = train.customer_id.map(num_cust_on_location)
test['num_cust_on_location'] = test.customer_id.map(num_cust_on_location)

In [42]:
indep_vars = ['location_number', 'latitude', 'longitude', 'vendor_id', 'vendor_latitude', 'vendor_longitude', 
            'delivery_charge', 'serving_distance', 'is_open', 'prepration_time', 'discount_percentage', 'status', 'verified', 
              'rank', 'vendor_rating', 'sunday_from_time1', 'sunday_to_time1', 'sunday_from_time2', 'sunday_to_time2', 
              'monday_from_time1', 'monday_to_time1', 'monday_from_time2', 'monday_to_time2', 'tuesday_from_time1', 
              'tuesday_to_time1', 'tuesday_from_time2', 'tuesday_to_time2', 'wednesday_from_time1', 'wednesday_to_time1',
              'wednesday_from_time2', 'wednesday_to_time2', 'thursday_from_time1', 'thursday_to_time1', 'thursday_from_time2', 
              'thursday_to_time2', 'friday_from_time1', 'friday_to_time1', 'friday_from_time2', 'friday_to_time2',
              'saturday_from_time1', 'saturday_to_time1', 'saturday_from_time2', 'saturday_to_time2', 'primary_tags', 
              'device_type', 'vendor_tags_svd_0', 'vendor_tags_svd_1', 'vendor_tags_svd_2', 'vendor_tags_svd_3',
              'vendor_tags_svd_4', 'vendor_tags_svd_5', 'vendor_tags_svd_6', 'vendor_tags_svd_7', 'vendor_member_since', 
              'customer_is_female', 'customer_age', 'customer_num_updates', 'member_since', 'location_cluster', 
              'vendor_location_cluster_order_count', 'longitude_corrected', 'latitude_corrected', 'vendor_longitude_corrected',
              'vendor_latitude_corrected', 'vendor_distance', 'vendor_distance_corrected', 'lon_diff', 'lat_diff', 
              'lon_diff_corrected', 'lat_diff_corrected', 'lon_diff_abs', 'lat_diff_abs', 'lon_diff_corrected_abs', 
              'lat_diff_corrected_abs', 'customer_location_counts', 'num_vendors_nearby', 'distance_rank', 'distance_rank_lat',
              'distance_rank_lon', 'vendor_uniuqe_customer_count', 'vendor_entropy', 'vendor_total_amount_by_location_type', 
              'daily_order_counts_mean', 'daily_order_counts_std', 'vendor_ratings_mean', 'vendor_ratings_std',
              'vendor_ratings_count', 'vendor_driver_ratings_mean', 'vendor_driver_ratings_std', 'vendor_driver_ratings_count',
              'vendor_is_restaurant', 'location_type_is_home', 'location_type_is_work', 'location_type_is_other', 
              'vendor_mean_delivery_distance', 'vendor_std_delivery_distance', 'vendor_max_delivery_distance', 
              'cust_location_lat_range', 'cust_location_lon_range', 'vendor_location_lat_range', 'vendor_location_lon_range',
              'member_since_diff', 'vendor_order_frequency', 'repeat_customers', 'vendor_latent_feature_0',
              'vendor_latent_feature_1', 'vendor_latent_feature_2', 'vendor_latent_feature_3', 'vendor_latent_feature_4', 
              'vendor_latent_feature_5', 'vendor_latent_feature_6', 'vendor_latent_feature_7', 'num_cust_on_location'
             ]
# indep_vars = [var for var in indep_vars if var not in vars_to_remove]
len(indep_vars)

113

In [ ]:
def lgb_f1_score(y_hat, data):
    return 'f1', f1_score(data.get_label(), y_hat > 0.125), True
params = {
    'task': 'train',
    'boosting_type': 'gbdt',
    'objective':'binary',
    'metric': {'auc'},
    'num_leaves': 256,
    'learning_rate': 0.01,
    'feature_fraction': 0.7,
    'bagging_fraction': 0.7,
    'bagging_freq': 1,
    'verbose': 1,
    'min_data_in_leaf' : 2,
    'max_bin' : 255,
    'lambda_l1' : 0.00002,
    'lambda_l2' : 0.00001,
    'min_gain_to_split' : 0.001
}
import random
random.seed(2)
models = []
for fold in sorted(validation_k_folds.fold.unique()):
    print("fold =", fold)
    dev_customers = set(train.customer_id) - set(validation_k_folds[validation_k_folds.fold == fold].customer_id)
    dev = train[train.customer_id.isin(dev_customers)]
    val = train[train.customer_id.isin(dev_customers) == False]

    print(dev.shape, val.shape)
#     lgb_dev = lgb.Dataset(dev[indep_vars], dev['target'])
#     lgb_val = lgb.Dataset(val[indep_vars], val['target'])

#     model = lgb.train(params, lgb_dev, num_boost_round = 3000, valid_sets = (lgb_val), early_stopping_rounds = 500,
#                          feval=lgb_f1_score, verbose_eval = 100)
    model = CatBoostClassifier(iterations=3000)

    # fit the model with the training data
    model.fit(dev[indep_vars],
        y=dev['target'],
        cat_features=None, 
        text_features=None,
        sample_weight=None,
        baseline=None,
        use_best_model=True,
        eval_set=(val[indep_vars], val['target']),
        verbose=100,
        logging_level=None,
        plot=False,
        column_description=None,
        verbose_eval=None,
        metric_period=None,
        silent=None,
        early_stopping_rounds=500,
        save_snapshot=None,
        snapshot_file=None,
        snapshot_interval=None,
        init_model=None)
              

    models.append(model)

fold = 1
(5107400, 133) (842900, 133)
Learning rate set to 0.161915
0:	learn: 0.4121194	test: 0.4120416	best: 0.4120416 (0)	total: 512ms	remaining: 25m 34s
100:	learn: 0.0518058	test: 0.0518011	best: 0.0518011 (100)	total: 50.6s	remaining: 24m 12s
200:	learn: 0.0506023	test: 0.0508657	best: 0.0508657 (200)	total: 1m 39s	remaining: 22m 59s
300:	learn: 0.0499156	test: 0.0504599	best: 0.0504599 (300)	total: 2m 26s	remaining: 21m 54s
400:	learn: 0.0493928	test: 0.0501649	best: 0.0501649 (400)	total: 3m 15s	remaining: 21m 6s
500:	learn: 0.0490550	test: 0.0500626	best: 0.0500626 (500)	total: 4m 3s	remaining: 20m 13s
600:	learn: 0.0487299	test: 0.0499697	best: 0.0499697 (600)	total: 4m 52s	remaining: 19m 26s
700:	learn: 0.0484538	test: 0.0499067	best: 0.0499055 (695)	total: 5m 40s	remaining: 18m 36s
800:	learn: 0.0482058	test: 0.0498618	best: 0.0498606 (799)	total: 6m 29s	remaining: 17m 49s
900:	learn: 0.0479454	test: 0.0497793	best: 0.0497793 (900)	total: 7m 17s	remaining: 16m 59s
1000:	lear

In [54]:
val1 = []
for i, fold in tqdm(enumerate(sorted(validation_k_folds.fold.unique()))):
    dev_customers = set(train.customer_id) - set(validation_k_folds[validation_k_folds.fold == fold].customer_id)
    val = train[train.customer_id.isin(dev_customers) == False]
    val['pred'] = models[i].predict_proba(val[indep_vars])[:,1]
    print(fold, f1_score(val.target, val.pred.map(lambda x : 1 if x > 0.125 else 0)))
    val1.append(val)
val1 = pd.concat(val1)   

/home/ubuntu/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """


1 0.25274764873337896
2 0.25380852550663874
3 0.25692588946702294
4 0.24956822107081175
5 0.2501159596103757
6 0.24509939260077307
7 0.25310899780541335



In [55]:
for i in range(1,11):
    th = 0.12 + i/1000
    print(th, f1_score(val1.target, val1.pred.map(lambda x : 1 if x > th else 0)))

0.121 0.25139610262818757
0.122 0.25158991120124435
0.123 0.2516462304176736
0.124 0.25160812716959613
0.125 0.25161736008511626
0.126 0.25168726622215
0.127 0.2515390209675602
0.128 0.2515316208083655
0.129 0.25157428929483283
0.13 0.25147360093052984


In [53]:
models[0].predict_proba(val[indep_vars])[:,1].mean()

0.013421786520530332

In [58]:
preds = []
for i in tqdm(range(5)):
    pred = models[i].predict_proba(test[indep_vars])[:,1]
    preds.append(pred)

In [59]:
test['pred'] = list(np.array(preds).mean(axis = 0))

In [60]:
test['target'] = test.pred.map(lambda x : 1 if x > 0.125 else 0)

In [64]:
test['CID X LOC_NUM X VENDOR'] = test.customer_id + " X " + test.location_number.astype(str) + " X " + test.vendor_id.astype(str)
test[['CID X LOC_NUM X VENDOR', 'target']].to_csv("./submission_{}.csv".format(version), index = False)
test[['CID X LOC_NUM X VENDOR', 'pred']].to_csv("./preds_{}.csv".format(version), index = False)